<a href="https://course.bioinfo.se/open.html?nb=session09-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#007ACC;color:#FFFFFF;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in VS Code</a>

# Session 9 lab: training networks yourself on blood-cell images {.unnumbered}

The Session 8 lab gave you a ready-made `train_model` function. In this lab
you **write the training loop yourself**, and then use it to run the
experiments behind the Session 9 page:

- which **loss function** works for a classifier;
- how **depth** and the choice of **activation** affect the gradient that
  reaches the first layer (vanishing gradients);
- **SGD vs. Adam**, and a wider hidden layer;
- a small **hyperparameter search**, chosen on the validation set, with
  the test set used exactly once.

An optional extra at the end (not part of the quiz) shows what happens
if every weight starts at the **same value**.

The data are the same as in the Session 8 lab: microscope images of single
blood cells from **BloodMNIST**, eight cell types, 28×28 colour pixels.

Every `todo("...")` call marks a piece of code for you to write: replace
the whole `todo(...)` call with your code. Work through the cells **in
order, top to bottom**. Until you fill it in, a cell stops with
`NotImplementedError: TODO in this cell: ...`, which tells you what is
missing. That is expected.

**Runtime:** a few minutes on Colab's free CPU. Keep `SEED = 0` so that
your numbers match the lab quiz. Neural-network training still varies a
little between machines, so the quiz accepts a range, and you upload
your notebook at the end.

In [ ]:
import os, tempfile, itertools, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))   # small networks gain nothing from more threads
device = "cpu"   # everything here is small enough for a CPU

def todo(what):
    """Placeholder for code you write: replace the whole todo(...) call with your own code."""
    raise NotImplementedError(f"TODO in this cell: {what}")

## 1. The data (as in the Session 8 lab)

**BloodMNIST** ([MedMNIST v2](https://medmnist.com/), Yang et al. 2023,
*Sci Data* 10:41, CC BY 4.0) holds light-microscope images of single blood
cells from healthy donors, resized to 28×28 colour pixels from the dataset
of Acevedo et al. 2020 (*Data Brief* 30:105474). There are eight classes:
basophil, eosinophil, erythroblast, immature granulocyte, lymphocyte,
monocyte, neutrophil and platelet, in their natural, unequal proportions.

The course uses a fixed subset of the official splits: 4,000 training,
1,000 validation and 2,000 test images. The test set stays untouched
until Section 6. Pixels are standardized per colour channel with
statistics from the **training images only** (Session 8), and each image
is flattened to a vector of 28 × 28 × 3 = 2,352 numbers.

`USE_FULL_DATA = True` instead downloads the complete dataset (35 MB,
official splits: 11,959 / 1,712 / 3,421 images). The quiz numbers refer
to the subset, so leave it at `False` for the quiz.

In [ ]:
USE_FULL_DATA = False

SUBSET_LOCAL = "data/bloodmnist-subset.npz"
SUBSET_URL = "https://raw.githubusercontent.com/ElofssonLab/kb8029-book/main/notebooks/data/bloodmnist-subset.npz"
FULL_URL = "https://zenodo.org/records/10519652/files/bloodmnist.npz?download=1"
CLASS_NAMES = ["basophil", "eosinophil", "erythroblast", "immature granulocyte",
               "lymphocyte", "monocyte", "neutrophil", "platelet"]

def fetch(url, name):
    path = os.path.join(tempfile.gettempdir(), name)
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    return path

if USE_FULL_DATA:
    data = np.load(fetch(FULL_URL, "bloodmnist.npz"))
elif os.path.exists(SUBSET_LOCAL):
    data = np.load(SUBSET_LOCAL)
else:
    data = np.load(fetch(SUBSET_URL, "bloodmnist-subset.npz"))
images = {s: data[f"{s}_images"].astype(np.float32) / 255.0 for s in ("train", "val", "test")}
labels = {s: data[f"{s}_labels"].reshape(-1).astype(np.int64) for s in ("train", "val", "test")}

mu = images["train"].mean(axis=(0, 1, 2))     # one mean per colour channel, training images only
sd = images["train"].std(axis=(0, 1, 2))
to_tensor = lambda X: torch.tensor(((X - mu) / sd).reshape(len(X), -1), dtype=torch.float32)
X_train, X_val, X_test = (to_tensor(images[s]) for s in ("train", "val", "test"))
y_train, y_val, y_test = (torch.tensor(labels[s]) for s in ("train", "val", "test"))
N_INPUTS, N_CLASSES = X_train.shape[1], len(CLASS_NAMES)
print("train:", tuple(X_train.shape), " val:", tuple(X_val.shape), " test:", tuple(X_test.shape))
print("validation images per class:", {c: int(n) for c, n in zip(CLASS_NAMES, np.bincount(labels["val"]))})

fig, axes = plt.subplots(1, 8, figsize=(12, 1.9))
for k, ax in enumerate(axes):
    ax.imshow(images["train"][np.where(labels["train"] == k)[0][0]])
    ax.set_title(CLASS_NAMES[k].replace(" ", "\n"), fontsize=8); ax.axis("off")
plt.show()

The classes are not balanced: neutrophils are the largest class, about
20% of the validation images. So a network that has learned nothing
except "always say neutrophil" already scores about **0.20**. That is the
real "chance level" to compare against here, not 1/8.

## 2. A network builder, and the training loop

`make_net` builds a fully connected network: `depth` hidden layers of
`width` units, each followed by an activation, and a final `Linear` layer
with 8 output logits, one per cell type. Fill in the one line that adds a
hidden layer.

In [ ]:
def make_net(depth=1, width=10, activation=nn.Tanh):
    layers, n_in = [], N_INPUTS
    for _ in range(depth):
        layers += todo('a list with nn.Linear(n_in, width) followed by activation()')
        n_in = width
    layers.append(nn.Linear(n_in, N_CLASSES))
    return nn.Sequential(*layers)

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()

def predicted_classes(model, X):
    """How many images the model assigns to each class."""
    model.eval()
    with torch.no_grad():
        return np.bincount(model(X).argmax(dim=1).numpy(), minlength=N_CLASSES)

print(make_net())

Now the heart of the lab: the **training loop**. A `DataLoader` shuffles
the training set every epoch and serves it in mini-batches. For every
batch, the same four steps from the Session 9 page:

1. clear the old gradients (`optimizer.zero_grad()`);
2. forward pass and loss (`loss = loss_fn(model(xb), yb)`);
3. backpropagation (`loss.backward()`);
4. update every weight (`optimizer.step()`).

After each epoch, record training and validation accuracy.

Some loss functions (MSE, MAE) compare the network's *probabilities* with
a **one-hot** target (a vector of eight numbers, 1 for the true cell type
and 0 elsewhere) instead of taking the class index directly. The argument
`one_hot=True` handles that: the loop then applies softmax to the logits
and compares with the one-hot vector.

The first network has 10 tanh hidden units and is trained with plain SGD,
learning rate 0.01, for 20 epochs.

In [ ]:
def train(model, loss_fn, optimizer, epochs=20, batch_size=64, one_hot=False, seed=SEED):
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    history = {"train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            # TODO: the four steps. Hint: if one_hot is True, the loss is
            #   loss_fn(torch.softmax(model(xb), dim=1), nn.functional.one_hot(yb, N_CLASSES).float())
            # otherwise it is loss_fn(model(xb), yb).
            todo('the four steps of the training loop (see the comment above)')
        history["train_acc"].append(accuracy(model, X_train, y_train))
        history["val_acc"].append(accuracy(model, X_val, y_val))
    return history

torch.manual_seed(SEED)
model = make_net()
hist = train(model, nn.CrossEntropyLoss(), torch.optim.SGD(model.parameters(), lr=0.01), epochs=20)
print(f"after 20 epochs: train accuracy {hist['train_acc'][-1]:.3f}, validation accuracy {hist['val_acc'][-1]:.3f}")

## 3. Which loss function?

Train the same network (10 tanh hidden units, SGD, learning rate 0.01, 20
epochs) three times, with three losses:

- **cross-entropy**, the standard choice for classification (Session 9 page);
- **mean squared error** between the softmax probabilities and the one-hot target;
- **mean absolute error** between the same two.

Re-create the model before each run, so that every run starts from
fresh weights. The last column counts how many of the eight cell types
the trained network ever predicts on the validation set.

In [ ]:
losses = {
    "cross-entropy": (nn.CrossEntropyLoss(), False),
    "MSE": (nn.MSELoss(), True),
    "MAE": (nn.L1Loss(), True),
}
loss_results = {}
for name, (loss_fn, one_hot) in losses.items():
    torch.manual_seed(SEED)
    m = make_net()
    h = train(m, loss_fn, torch.optim.SGD(m.parameters(), lr=0.01), epochs=20, one_hot=one_hot)
    loss_results[name] = h
    print(f"{name:14s} validation accuracy after 20 epochs: {h['val_acc'][-1]:.3f}   "
          f"classes predicted: {(predicted_classes(m, X_val) > 0).sum()} of {N_CLASSES}")

plt.figure(figsize=(6, 3.5))
for name, h in loss_results.items():
    plt.plot(range(1, len(h["val_acc"]) + 1), h["val_acc"], label=name)
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend(); plt.show()

## 4. Depth and vanishing gradients

Build networks with 1, 4 and 8 hidden layers of 10 units, with tanh,
sigmoid and ReLU activations. For each, first measure the mean absolute
**gradient of the loss with respect to the first layer's weights** at
initialization, on the whole training set. Then train for 10 epochs and
record validation accuracy. This section uses a ten times larger
learning rate (SGD, 0.1), to give the deep networks a fair chance of
getting somewhere in 10 epochs.

In [ ]:
depth_results = {}
for activation in [nn.Tanh, nn.Sigmoid, nn.ReLU]:
    for depth in [1, 4, 8]:
        torch.manual_seed(SEED)
        m = make_net(depth=depth, activation=activation)
        loss = nn.CrossEntropyLoss()(m(X_train), y_train)
        first_layer_grad = todo('gradient of loss w.r.t. m[0].weight (hint: torch.autograd.grad(...)[0])')
        grad_size = first_layer_grad.abs().mean().item()
        h = train(m, nn.CrossEntropyLoss(), torch.optim.SGD(m.parameters(), lr=0.1), epochs=10)
        n_predicted = (predicted_classes(m, X_val) > 0).sum()
        depth_results[(activation.__name__, depth)] = (grad_size, h["val_acc"][-1])
        print(f"{activation.__name__:8s} depth {depth}: first-layer |gradient| {grad_size:.2e}   "
              f"validation accuracy {h['val_acc'][-1]:.3f}   classes predicted {n_predicted}")

ratio = depth_results[("Sigmoid", 1)][0] / depth_results[("Sigmoid", 8)][0]
print(f"\nsigmoid: first-layer gradient with 1 hidden layer / with 8 hidden layers = {ratio:.1e}")

## 5. SGD, momentum and Adam, and a wider hidden layer

Go back to one hidden layer, but make it 128 units wide. Train it for 10
epochs three times: with plain SGD (learning rate 0.01), with SGD plus
**momentum** 0.9 (same learning rate), and with **Adam** (learning rate
0.001). The section *Momentum and Adam* on the Session 9 page explains
what the two additions do.

In [ ]:
wide = make_net(depth=1, width=128)
n_params_wide = todo('total number of trainable parameters')
print("parameters in the 128-unit network:", n_params_wide)

optimizer_results = {}
for name in ["SGD", "SGD + momentum", "Adam"]:
    torch.manual_seed(SEED)
    m = make_net(depth=1, width=128)
    if name == "SGD":
        opt = torch.optim.SGD(m.parameters(), lr=0.01)
    elif name == "SGD + momentum":
        opt = todo('SGD with learning rate 0.01 and momentum 0.9')
    else:
        opt = todo('an Adam optimizer with learning rate 0.001')
    h = train(m, nn.CrossEntropyLoss(), opt, epochs=10)
    optimizer_results[name] = h
    print(f"{name:14s}: train accuracy {h['train_acc'][-1]:.3f}, validation accuracy {h['val_acc'][-1]:.3f}")

plt.figure(figsize=(6, 3.5))
for name, h in optimizer_results.items():
    plt.plot(range(1, len(h["val_acc"]) + 1), h["val_acc"], label=name)
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend(); plt.show()

**Explain your result.** Which optimizer got furthest in 10 epochs? Use
the two ideas from the Session 9 page: momentum averages recent gradients
(so steps that agree add up and noisy ones cancel), and Adam also gives
every weight a step of about the learning rate, whatever the size of its
gradient. Would the ranking necessarily stay the same with more epochs, or
with other learning rates? (Canvas asks about this.)

Does the hidden layer pay off here? The Session 9 page compared every
network against a **linear** baseline. `make_net(depth=0)` has no hidden
layer at all: a single `Linear(2352, 8)`, i.e. multi-class logistic
regression. Train it the same way (Adam, learning rate 0.001, 10 epochs).

In [ ]:
torch.manual_seed(SEED)
linear = todo('a network with no hidden layer')
h_lin = train(linear, nn.CrossEntropyLoss(), torch.optim.Adam(linear.parameters(), lr=1e-3), epochs=10)
n_params_linear = sum(p.numel() for p in linear.parameters())
print(f"linear model: {n_params_linear} parameters, validation accuracy {h_lin['val_acc'][-1]:.3f}")
print(f"128-unit network with Adam: validation accuracy {optimizer_results['Adam']['val_acc'][-1]:.3f}")
print(f"training images per weight in the 128-unit network: {len(X_train) / n_params_wide:.3f}")

## 6. A small hyperparameter search, then the test set once

Try every combination of hidden-layer width (32 or 128) and dropout rate
(0 or 0.3) after the hidden layer, with Adam (learning rate 0.001, 10
epochs). **Choose the best combination on the validation set.** Only
then evaluate that one model on the test set, once. Choosing on the test
set would make the test number optimistic (Session 8).

In [ ]:
def make_dropout_net(width, dropout):
    return nn.Sequential(nn.Linear(N_INPUTS, width), nn.ReLU(), nn.Dropout(dropout), nn.Linear(width, N_CLASSES))

search = {}
for width, dropout in itertools.product([32, 128], [0.0, 0.3]):
    torch.manual_seed(SEED)
    m = make_dropout_net(width, dropout)
    h = train(m, nn.CrossEntropyLoss(), torch.optim.Adam(m.parameters(), lr=1e-3), epochs=10)
    search[(width, dropout)] = (h["val_acc"][-1], m)
    print(f"width {width:3d}, dropout {dropout}: validation accuracy {h['val_acc'][-1]:.3f}")

best_config = todo('the (width, dropout) with the highest VALIDATION accuracy')
best_model = search[best_config][1]
test_accuracy = todo('accuracy of best_model on the test set (used once, here)')
print(f"\nchosen on validation: width {best_config[0]}, dropout {best_config[1]} "
      f"(validation accuracy {search[best_config][0]:.3f})")
print(f"test accuracy of that model: {test_accuracy:.3f}")

**Last step:** save your notebook with all outputs (File → Download →
.ipynb) and upload it with the lab quiz on Canvas. If one of your numbers
falls outside the quiz's accepted range, the notebook lets us see whether
that is ordinary run-to-run variation in training.

The optional extra below is not part of the quiz.

## Optional extra (not part of the quiz): what if every weight starts at 1?

Take the same network, but before training set **every** weight and bias
to 1. Then train as before (SGD, learning rate 0.01, 20 epochs) and look
at the hidden layer's weights: the 10 rows of `model[0].weight`, one row
of 2,352 weights per hidden unit. Compare the differences *between* the
units with how far the weights moved from 1 during training.

In [ ]:
torch.manual_seed(SEED)
ones_model = make_net()
with torch.no_grad():
    for p in ones_model.parameters():
        todo('set every entry of p to 1.0 (hint: p.fill_)')
h_ones = train(ones_model, nn.CrossEntropyLoss(), torch.optim.SGD(ones_model.parameters(), lr=0.01), epochs=20)
W = ones_model[0].weight.detach()
max_row_difference = todo("largest absolute difference between any hidden unit's weights and unit 0's")
print(f"validation accuracy: {h_ones['val_acc'][-1]:.3f}")
print(f"largest difference between hidden units' weights after training: {max_row_difference:.2e}")
print(f"largest change of any weight from its starting value 1:          {(W - 1).abs().max().item():.2e}")
counts = predicted_classes(ones_model, X_val)
print("validation images assigned to each class:", {c: int(n) for c, n in zip(CLASS_NAMES, counts)})